## Ingestão de Dados (Camada Bronze) — Abordagem ELT Pura e Resiliente

Este script implementa a etapa de **Extração e Carga (EL)** do pipeline utilizando Python nativo (`requests` e `pathlib`), seguindo as diretrizes modernas de arquitetura de Data Lakehouse para a Camada Bronze.

### Objetivos da Arquitetura:
1. **ELT Legítimo (Imutabilidade):** O dado bruto é extraído da fonte e carregado na Bronze exatamente em seu formato original, sem sofrer nenhuma transformação ou conversão nesta etapa.
2. **Eficiência Computacional:** Evita o uso do Spark para leitura/escrita na ingestão, eliminando o custo desnecessário de processamento (*overhead*) e o risco do `inferSchema` em dados brutos.
3. **Mecanismo de Cache Nativo:** O script valida se o arquivo já existe localmente antes de iniciar a requisição, poupando banda de rede e garantindo a idempotência do pipeline.

### Mecanismos de Resiliência Implementados:
* **Streaming em Chunks (1MB):** O download é processado em blocos de 1MB, mantendo o consumo de memória RAM fixo e minimalista, independente do tamanho do arquivo (escalabilidade).
* **Tratamento de Erros HTTP & Timeout:** Garante a segurança da transmissão validando o Status 200 e aplicando um limite de tempo (`timeout=30`) para evitar travamentos infinitos.
* **Escrita Atômica (`.tmp`):** O arquivo é gravado com uma extensão temporária. Se o download for interrompido por oscilações de rede, o arquivo parcial é deletado automaticamente (`.unlink()`). O arquivo oficial `.parquet` só passa a existir quando está 100% íntegro no disco, blindando o ecossistema contra dados corrompidos.

In [11]:
import os
import requests
from pathlib import Path

url_parquet = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

diretorio_raiz = Path(os.getcwd())
diretorio_bronze = diretorio_raiz / "data" / "1_bronze" / "yellow_trips"
diretorio_bronze.mkdir(parents=True, exist_ok=True)

arquivo_destino = diretorio_bronze / "yellow_tripdata_2024-01.parquet"
# Criamos um arquivo temporário de download na mesma pasta
arquivo_tmp = diretorio_bronze / "yellow_tripdata_2024-01.parquet.tmp"

# Só inicia se o arquivo final e definitivo NÃO existir
if not arquivo_destino.exists():
    try:
        response = requests.get(url_parquet, stream=True, timeout=30)

        if response.status_code == 200:
            chunk_size_1mb = 1024 * 1024
            with open(arquivo_tmp, "wb") as f:
                for chunk in response.iter_content(chunk_size=chunk_size_1mb):
                    if chunk: # Garante que só escreve se o chunk contiver bytes
                        f.write(chunk)

            # Se o laço for chegou aqui sem erros, o arquivo está completo.
            # Agora renomeamos o .tmp para o arquivo oficial .parquet
            arquivo_tmp.rename(arquivo_destino)
            print("Download do CSV concluído com sucesso e salvo na Camada Bronze!")

        else:
            raise Exception(f"Erro HTTP: {response.status_code}")

    except Exception as e:
        # SE FALHAR NO MEIO: Se caiu a internet no meio do 'for', deletamos o arquivo temporário incompleto
        if arquivo_tmp.exists():
            arquivo_tmp.unlink()
        raise Exception(f"O download falhou no meio do processo ou a rede caiu: {e}")

Download do CSV concluído com sucesso e salvo na Camada Bronze!


**Extraindo e Carregando a tabela de Taxi Zone Bruta**

In [12]:
url_csv = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"


diretorio_raiz = Path(os.getcwd())
diretorio_bronze = diretorio_raiz / "data" / "1_bronze" / "yellow_trips"
diretorio_bronze.mkdir(parents=True, exist_ok=True)


arquivo_destino_csv = diretorio_bronze / "taxi_zone_lookup.csv"
arquivo_tmp_csv = diretorio_bronze / "taxi_zone_lookup.csv.tmp"

if not arquivo_destino_csv.exists():
    try:

        response = requests.get(url_csv, stream=True, timeout=30)

        if response.status_code == 200:
            chunk_size_1mb = 1024 * 1024


            with open(arquivo_tmp_csv, "wb") as f:
                for chunk in response.iter_content(chunk_size=chunk_size_1mb):
                    if chunk:
                        f.write(chunk)

            # Se o laço for chegou aqui sem erros, o arquivo está completo.
            # Agora renomeamos o .tmp para o arquivo oficial .parquet
            arquivo_tmp_csv.rename(arquivo_destino_csv)
            print("Download do CSV concluído com sucesso e salvo na Camada Bronze!")

        else:
            raise Exception(f"Erro HTTP ao tentar acessar o CSV: {response.status_code}")

    except Exception as e:
        # SE FALHAR NO MEIO: Se caiu a internet no meio do 'for', deletamos o arquivo temporário incompleto
        if arquivo_tmp_csv.exists():
            arquivo_tmp_csv.unlink()
        raise Exception(f"O download do CSV falhou ou foi interrompido: {e}")
else:
    print("O arquivo CSV já existe na Camada Bronze. Download ignorado (Cache ativo).")

Download do CSV concluído com sucesso e salvo na Camada Bronze!
